In [ ]:
from huggingface_hub import login
import os
import sys
import csv
from tqdm import trange
from transformers import AutoModel,AutoTokenizer
FILE_PATH = './QA_results_GT.csv'
os.environ["OPENAI_API_KEY"] = "your key"
os.environ["OPENAI_API_BASE"] = "https://api.openai.com/v1"
import pandas as pd
import os

In [ ]:
ANA_FILE_PATH = 'your path/GPT51_output.csv'

# Only keep: Question, Gold Answer, MiniRAG answer
minianswer_LIST = []
QUESTION_LIST = []
GA_LIST = []
filelength = 0
with open(ANA_FILE_PATH, mode='r', encoding='utf-8') as question_file:
    reader = csv.DictReader(question_file)
    for row in reader:
        QUESTION_LIST.append(row['Question'])
        GA_LIST.append(row['Gold Answer'])
        # Comment out other systems; only keep MiniRAG
        # naiveanswer_LIST.append(row['naive'])
        # lightraganswer_LIST.append(row['lightrag'])
        minianswer_LIST.append(row['minirag'])
        filelength = filelength+1

In [21]:
PROMPT = """
Now, I'll give you a question, a gold answer to this question, and one answer provided by a student.

Determine the answer according to the following rules:
If the answer is correct, get 1 point.
If the answer is irrelevant to the question, it will receive 0 points.
If the answer is incorrect, get -1 point.

Return your answer in JSON mode. Use the key "Score".

For example:

Question:
When does Li Hua arrive to the city?

Gold Answer:
20260105

Answer: LiHua arrived on the afternoon of January 5th

output:
{{
"Score": 1
}}


Real data:

Question:
{question}
Gold Answer:
{ga}

Answer: {mini}

output:

"""

In [22]:
# openai via custom proxy gateway (set OPENAI_BASE_URL and OPENAI_API_KEY in env)
from openai import OpenAI
from tqdm import trange
import os, time

BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")  # your proxy, override via env
API_KEY = os.getenv("OPENAI_API_KEY")  # set this in your shell

chatbot = OpenAI(api_key=API_KEY, base_url=BASE_URL)

def call_with_retry(prompt: str, retries: int = 5, backoff: float = 1.5) -> str:
    for i in range(retries):
        try:
            resp = chatbot.chat.completions.create(
                messages=[{"role": "system", "content": prompt}],
                model="gpt-4o",
            )
            return resp.choices[0].message.content.strip()
        except Exception as e:
            if i == retries - 1:
                raise
            time.sleep(backoff * (2 ** i))

chat_list = []
for i in trange(filelength):
    p = PROMPT.format(question=QUESTION_LIST[i], ga=GA_LIST[i], mini=minianswer_LIST[i])
    chat_list.append(call_with_retry(p))


100%|██████████| 86/86 [01:05<00:00,  1.32it/s]


In [23]:
import json
import json_repair

# Parse LLM outputs
chat_score_list = []
for chat in chat_list:
    try:
        data = json_repair.loads(chat.strip('```json').strip('```'))
        chat_score_list.append(data)
    except Exception:
        chat_score_list.append({"Score": 0})
        print('Error in chat:', chat)

# Only one score now
all_scores = [data.get('Score', 0) for data in chat_score_list]

num_all = len(all_scores)
num_pos = all_scores.count(1)
num_zero = all_scores.count(0)
num_neg = all_scores.count(-1)

print(f"MiniRAG Score 1: {num_pos / num_all * 100:.2f}\%, Score 0: {num_zero / num_all * 100:.2f}\%, Score -1: {num_neg / num_all * 100:.2f}\%")

MiniRAG Score 1: 90.70\%, Score 0: 2.33\%, Score -1: 6.98\%


In [24]:
import pandas as pd

# Map numeric score to human-readable label
def score_to_label(score):
    if score == 1:
        return "Correct"
    elif score == 0:
        return "Unclear"
    elif score == -1:
        return "Wrong"
    else:
        return "Invalid"

# Build output DataFrame
df_out = pd.DataFrame({
    "Question": QUESTION_LIST,
    "GroundTruth": GA_LIST,
    "MiniAnswer": minianswer_LIST,
    "GPT_Output": chat_list,
    "Score": all_scores,
})

# Add correctness label
df_out["Correctness"] = df_out["Score"].apply(score_to_label)

# Save to CSV
df_out.to_csv("GPT51_output.csv", index=False)

print("Saved to GPT51_output.csv with Correctness column.")


Saved to GPT51_output.csv with Correctness column.


In [ ]:
# set directory
data_dir = r"your path"

In [ ]:
# read GPT51_output.xlsx
GPT_output = pd.read_excel(os.path.join(data_dir, 'GPT51_output.xlsx'))
GPT_output

,Question,GroundTruth,MiniAnswer,Correctness,Evidence,Type,ques_reason,Origin_evalu,Label_change_reason
0,Did Li Hua send a message to Jennifer thanking...,Yes,### Short answer Yes. Li Hua *did* send Jenni...,Correct,20260204_15:00<and>20260204_16:00<and>20260211...,Multi,NaN,NaN,NaN
1,Did Yuriko ask Li Hua for help with her studio...,Yes,Yes. Yuriko asked Li Hua for help with her stu...,Correct,20260223_15:00<and>20260225_15:00,Multi,NaN,NaN,NaN
2,Did Li Hua send a message to Jennifer asking i...,Yes,"## AnswerYes. Based on the messages provided, ...",Correct,20260211_19:00<and>20260309_12:00,Multi,NaN,NaN,NaN
3,"Who does Li Hua go to watch the movie ""Overwat...",Wolfgang,"## AnswerLi Hua goes to watch the movie **""Ove...",Correct,20260122_17:00<and>20260121_13:00,Multi,NaN,NaN,NaN
4,Who wished Li Hua a happy Lunar New Year?,Adam Smith & Jennifer Moore & Wolfgang Schulz,"From the provided messages, the following peop...",Correct,20260119_11:30<and>20260119_14:30<and>20260119...,Multi,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
81,Did Li Hua's complaint about the customer who ...,No,## AnswerYes. Li Hua’s complaint about the cli...,Wrong,20260123_17:30<and>20260131_14:00,Multi,Fail to extract the key/precise information,NaN,NaN
82,Did Yuriko tell Li Hua about booking a seat at...,Yes,"No. Based on the messages, Yuriko did **not** ...",Wrong,20260225_15:00<and>20260303_09:30,Multi,Fail to extract the key/precise information,NaN,NaN
83,"What time does Li Hua watch the movie ""Overwat...",20260122,## AnswerLi Hua is planning to watch the movie...,Correct,20260122_17:00<and>20260121_13:00,Multi,NaN,Wrong,Time can be date or clock
84,What time does Li Hua check in with Adam about...,5:30 PM,Li Hua checks in with Adam about moving in at ...,Correct,20260105_14:00,Single,Time information,NaN,NaN


In [4]:
# show how many Correctness are not 'Correct' by Type
print(GPT_output[GPT_output['Correctness'] != 'Correct']['Type'].value_counts())
# show the distribution of Type
print(GPT_output['Type'].value_counts())

# show the percentile of Single Type questions are not Correct and how many Multi Type questions are not Correct
single_incorrect = GPT_output[(GPT_output['Type'] == 'Single') & (GPT_output['Correctness'] != 'Correct')]
multi_incorrect = GPT_output[(GPT_output['Type'] == 'Multi') & (GPT_output['Correctness'] != 'Correct')]
print(f"Percentile of Single Type questions not Correct: {len(single_incorrect)/len(GPT_output[GPT_output['Type'] == 'Single'])*100:.2f}%")
print(f"Percentile of Multi Type questions not Correct: {len(multi_incorrect)/len(GPT_output[GPT_output['Type'] == 'Multi'])*100:.2f}%")

Type
Multi     3
Single    2
Name: count, dtype: int64
Type
Single    69
Multi     17
Name: count, dtype: int64
Percentile of Single Type questions not Correct: 2.90%
Percentile of Multi Type questions not Correct: 17.65%


In [5]:
# show the percentile distribution of Correctness
print(GPT_output['Correctness'].value_counts(normalize=True) * 100)
# merge Correctness value to Origin_evalu if Origin_evalu is NaN
GPT_output['Origin_evalu'] = GPT_output['Origin_evalu'].fillna(GPT_output['Correctness'])
# show the percentile distribution of Origin_evalu
print(GPT_output['Origin_evalu'].value_counts(normalize=True) * 100)

Correctness
Correct    94.186047
Wrong       4.651163
Unclear     1.162791
Name: proportion, dtype: float64
Origin_evalu
Correct    90.697674
Wrong       6.976744
Unclear     2.325581
Name: proportion, dtype: float64


In [6]:
# show the value distribution of ques_reason and to a df
ques_reason_df = GPT_output['ques_reason'].value_counts().reset_index()
ques_reason_df.columns = ['ques_reason', 'count']
ques_reason_df

,ques_reason,count
0,Fail to extract the key/precise information,4
1,Time information,2
